# 01 · Extração: SQL Server → MinIO (`landing-zone`, CSV)

Extrai **todas as tabelas** do banco `AcademiaDB` e grava cada uma como um arquivo CSV no bucket
`landing-zone` do MinIO. A landing zone guarda o dado **exatamente como veio da fonte**, sem tratamento.

**Antes de rodar:** notebook `00` executado.

## 1. Configuração

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Raiz do projeto (funciona abrindo o notebook pelo VS Code, pelo JupyterLab ou pelo nbconvert)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / ".env").exists():
    raise FileNotFoundError("Arquivo .env não encontrado. Na raiz do projeto rode: cp .env.example .env")
load_dotenv(ROOT / ".env", override=True)

DB_SERVER   = os.getenv("DB_SERVER")
DB_PORT     = os.getenv("DB_PORT")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_DATABASE = os.getenv("DB_DATABASE")

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
LANDING_BUCKET   = os.getenv("MINIO_LANDING_BUCKET")
BRONZE_BUCKET    = os.getenv("MINIO_BRONZE_BUCKET")

print(f"SQL Server: {DB_SERVER}:{DB_PORT}/{DB_DATABASE}")
print(f"MinIO: {MINIO_ENDPOINT} | bucket de destino: {LANDING_BUCKET}")

In [ ]:
import time
import pyodbc


def conectar(database="master", tentativas=12):
    """Conecta ao SQL Server, tentando de novo enquanto o container ainda está iniciando."""
    conn_str = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={DB_SERVER},{DB_PORT};DATABASE={database};"
        f"UID={DB_USER};PWD={DB_PASSWORD};TrustServerCertificate=yes;"
    )
    for i in range(1, tentativas + 1):
        try:
            return pyodbc.connect(conn_str, autocommit=True, timeout=5)
        except pyodbc.Error as erro:
            print(f"Tentativa {i}/{tentativas}: SQL Server ainda não respondeu ({erro.args[0]}). Aguardando 5 s...")
            time.sleep(5)
    raise RuntimeError("Não foi possível conectar ao SQL Server. Confira com: docker compose ps")

## 2. Listar as tabelas do banco

A lista vem do próprio SQL Server (`INFORMATION_SCHEMA.TABLES`), e não de uma lista fixa:
se uma tabela nova for criada no banco, ela também é extraída.

In [ ]:
conn = conectar(DB_DATABASE)
cursor = conn.cursor()

cursor.execute("""
    SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE' AND TABLE_SCHEMA = 'dbo'
    ORDER BY TABLE_NAME
""")
tabelas = [linha[0] for linha in cursor.fetchall()]

print(f"{len(tabelas)} tabelas encontradas:")
for t in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM dbo.[{t}]")
    print(f"  - {t:<12} {cursor.fetchone()[0]:>6} registros")

## 3. Criar o bucket `landing-zone` no MinIO

In [ ]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
    region_name="us-east-1",
)


def garantir_bucket(nome):
    """Cria o bucket no MinIO se ele ainda não existir."""
    existentes = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    if nome in existentes:
        print(f"Bucket [{nome}] já existe")
    else:
        s3.create_bucket(Bucket=nome)
        print(f"Bucket [{nome}] criado!")

garantir_bucket(LANDING_BUCKET)

## 4. Extrair cada tabela para CSV no MinIO

Para cada tabela: `SELECT *` no SQL Server → DataFrame do pandas → texto CSV na memória → `put_object` no MinIO.
O arquivo não passa pelo disco do computador.

In [ ]:
import io
import pandas as pd

resultados = []
for tabela in tabelas:
    cursor.execute(f"SELECT * FROM dbo.[{tabela}]")
    colunas = [d[0] for d in cursor.description]
    df = pd.DataFrame.from_records([tuple(l) for l in cursor.fetchall()], columns=colunas)

    buffer = io.StringIO()
    df.to_csv(buffer, index=False)
    conteudo = buffer.getvalue().encode("utf-8")

    chave = f"{tabela}.csv"
    s3.put_object(Bucket=LANDING_BUCKET, Key=chave, Body=conteudo, ContentType="text/csv")
    resultados.append({"tabela": tabela, "registros": len(df), "colunas": len(colunas),
                       "tamanho_kb": round(len(conteudo) / 1024, 1)})
    print(f"  {tabela:<12} -> s3://{LANDING_BUCKET}/{chave}  ({len(df)} registros)")

## 5. Validação

In [ ]:
objetos = {o["Key"]: o["Size"] for o in s3.list_objects_v2(Bucket=LANDING_BUCKET).get("Contents", [])}
print(f"Arquivos no bucket [{LANDING_BUCKET}]:")
for chave, tamanho in sorted(objetos.items()):
    print(f"  {chave:<18} {tamanho / 1024:>8.1f} KB")

for r in resultados:
    assert f"{r['tabela']}.csv" in objetos, f"{r['tabela']}.csv não está no MinIO"
print(f"\nOK: {len(resultados)} tabelas extraídas, {sum(r['registros'] for r in resultados):,} registros no total.")
pd.DataFrame(resultados)

In [ ]:
# Amostra do arquivo como ele ficou no MinIO (primeiras linhas do alunos.csv)
corpo = s3.get_object(Bucket=LANDING_BUCKET, Key="alunos.csv")["Body"].read().decode("utf-8")
print("\n".join(corpo.splitlines()[:6]))

In [ ]:
cursor.close()
conn.close()
print("Conexão encerrada.")